# 03. Step 1 to Step 2 Pipeline Bridge: Candidate Pair Generation

**Aspect-Category-Opinion-Sentiment (ACOS) Quadruple Extraction**

This notebook serves as the **pipeline bridge** between Step 1 (Co-Extraction) and Step 2 (Classification):
- Reads predicted aspect and opinion tags from `pred4pipeline.txt` generated in Step 1.
- Generates Cartesian combinations $(a, o)$ combining detected aspect spans and opinion spans (including `[-1, -1]` for implicit entities).
- Produces the formatted TSV dataset `[domain]_test_pair_1st.tsv` (`text####asp_span opi_span`) needed by Step 2 for pipeline evaluation.
- Compares candidate pairs yield, implicit/explicit distribution, and recall against Ground Truth pairs.
- Exports `candidate_pairs_summary.csv` and visualization charts.

## 1. Environment & Path Setup

In [ ]:
!pip install -q pytorch-crf transformers huggingface_hub seaborn scikit-learn matplotlib pandas boto3
import os
import sys
import codecs as cs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Detect if repository is present; if running in fresh Colab session, auto-clone repository
if not os.path.exists("Extract-Classify-ACOS") and not os.path.exists("../Extract-Classify-ACOS"):
    if not os.path.exists("ACOS"):
        print("📥 Cloning ACOS repository from GitHub into Colab environment...")
        !git clone https://github.com/haisyamalawwab/ACOS.git

# 2. Robustly locate base project directory across Colab & Local
if os.path.exists("Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath(".")
elif os.path.exists("../Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("..")
elif os.path.exists("ACOS/Extract-Classify-ACOS"):
    base_project_dir = os.path.abspath("ACOS")
elif os.path.exists("/content/ACOS/Extract-Classify-ACOS"):
    base_project_dir = "/content/ACOS"
elif os.path.exists("/content/Extract-Classify-ACOS"):
    base_project_dir = "/content"
else:
    base_project_dir = os.path.abspath(".")

extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
notebooks_dir = os.path.join(base_project_dir, "notebooks")

for p in [base_project_dir, extract_dir, notebooks_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

# 3. Import colab_utils with fallback download
try:
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )
except ModuleNotFoundError:
    import urllib.request
    print("⚠️ Downloading colab_utils.py fallback directly from GitHub...")
    raw_url = "https://raw.githubusercontent.com/haisyamalawwab/ACOS/main/notebooks/colab_utils.py"
    urllib.request.urlretrieve(raw_url, "colab_utils.py")
    from colab_utils import (
        setup_timestamped_run_dir, download_bert_pretrained, analyze_and_plot_eda,
        plot_training_history, export_benchmark_tables_and_plots,
        display_quadruple_dataframe, df_to_markdown, export_step_table,
        MarkdownReport, SubtaskMetricCapture, plot_subtask_metrics,
        features_step1, features_step2, pair_examples_from_file,
        resolve_eval_pair_file, unpack_model_output,
    )

DOMAIN = "rest16"   # 'rest16' or 'laptop'

# Locate most recent timestamped session folder or specify path
results_base = os.path.join(base_project_dir, "results")
session_folders = sorted([f for f in os.listdir(results_base) if f.startswith(DOMAIN)]) if os.path.exists(results_base) else []

if session_folders:
    active_session_dir = os.path.join(results_base, session_folders[-1])
    print(f"📂 Using latest session directory: {active_session_dir}")
else:
    dirs = setup_timestamped_run_dir(base_dir=results_base, domain=DOMAIN)
    active_session_dir = dirs["root"]

logs_dir = os.path.join(active_session_dir, "logs")
csv_dir = os.path.join(active_session_dir, "csv")
plots_dir = os.path.join(active_session_dir, "plots")
os.makedirs(logs_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

print(f"📂 Base project directory: {base_project_dir}")
print(f"📁 Extract & Model directory: {extract_dir}")

md_dir = os.path.join(active_session_dir, "md")
os.makedirs(md_dir, exist_ok=True)

session_dirs = {"root": active_session_dir, "logs": logs_dir, "csv": csv_dir,
                "plots": plots_dir, "md": md_dir}

rep = MarkdownReport(
    f"03 - Jembatan Step 1 ke Step 2: Pasangan Kandidat [{DOMAIN.upper()}]",
    md_dir,
    filename="03_pasangan_kandidat.md",
    meta={"domain": DOMAIN, "session_dir": active_session_dir},
)
print(f"[md] Hasil teks notebook ini ditulis ke: {md_dir}")


## 2. Locate Step 1 Predictions (`pred4pipeline.txt`)
Search in the active session logs or fallback to pre-generated predictions.

In [ ]:
candidate_pred_files = [
    os.path.join(logs_dir, "pred4pipeline.txt"),
    os.path.join(results_base, f"{DOMAIN}_1st", "pred4pipeline.txt"),
    os.path.join(extract_dir, "output", "Extract-Classify-QUAD", f"{DOMAIN}_1st", "pred4pipeline.txt"),
    os.path.join(extract_dir, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")
]

pred_file = None
for p in candidate_pred_files:
    if os.path.exists(p) and p.endswith("pred4pipeline.txt"):
        pred_file = p
        break

if pred_file:
    print(f"✅ Found Step 1 prediction file: {pred_file}")
else:
    print("ℹ️ No active 'pred4pipeline.txt' found in logs. Checking tokenized_data...")
    tokenized_pair = os.path.join(extract_dir, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")
    if os.path.exists(tokenized_pair):
        print(f"✅ Existing pre-computed test pairs found at: {tokenized_pair}")

## 3. Generate Candidate Aspect-Opinion Pairs
Parse predictions, handle implicit entities `[-1, -1]`, build Cartesian pairs, and write output files.

In [ ]:
# Target output files
target_tokenized_tsv = os.path.join(extract_dir, "tokenized_data", f"{DOMAIN}_test_pair_1st.tsv")
session_tsv_copy = os.path.join(logs_dir, f"{DOMAIN}_test_pair_1st.tsv")

pair_records = []

if pred_file and os.path.exists(pred_file):
    with cs.open(pred_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    with cs.open(target_tokenized_tsv, 'w', encoding='utf-8') as wf, \
         cs.open(session_tsv_copy, 'w', encoding='utf-8') as sf:
         
        for idx, line in enumerate(lines):
            asp = []
            opi = []
            line = line.strip().split('\t')
            if len(line) <= 1:
                continue
            text = line[0]
            af = 0
            of = 0
            for ele in line[1:]:
                if ele.startswith('a'):
                    asp.append(ele[2:])
                    af = 1
                else:
                    opi.append(ele[2:])
                    of = 1
            if af == 0:
                asp.append('-1,-1')
            if of == 0:
                opi.append('-1,-1')
                
            for pa in asp:
                for po in opi:
                    out_line = f"{text}####{pa} {po}\n"
                    wf.write(out_line)
                    sf.write(out_line)
                    
                    pair_records.append({
                        "Sentence_ID": idx,
                        "Text": text,
                        "Aspect_Span": pa,
                        "Opinion_Span": po,
                        "Is_Implicit_Aspect": (pa == "-1,-1"),
                        "Is_Implicit_Opinion": (po == "-1,-1"),
                        "Pair_Type": f"{'Implicit' if pa=='-1,-1' else 'Explicit'}-{'Implicit' if po=='-1,-1' else 'Explicit'}"
                    })
                    
    print(f"✅ Successfully generated {len(pair_records)} candidate pairs.")
    print(f"   - Saved to: {target_tokenized_tsv}")
    print(f"   - Saved to: {session_tsv_copy}")
else:
    # Load existing pairs for analysis
    if os.path.exists(target_tokenized_tsv):
        with open(target_tokenized_tsv, 'r', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                parts = line.strip().split("####")
                if len(parts) == 2:
                    text = parts[0]
                    spans = parts[1].split(" ")
                    pa = spans[0] if len(spans) > 0 else "-1,-1"
                    po = spans[1] if len(spans) > 1 else "-1,-1"
                    pair_records.append({
                        "Sentence_ID": idx,
                        "Text": text,
                        "Aspect_Span": pa,
                        "Opinion_Span": po,
                        "Is_Implicit_Aspect": (pa == "-1,-1"),
                        "Is_Implicit_Opinion": (po == "-1,-1"),
                        "Pair_Type": f"{'Implicit' if pa=='-1,-1' else 'Explicit'}-{'Implicit' if po=='-1,-1' else 'Explicit'}"
                    })
        print(f"ℹ️ Loaded {len(pair_records)} existing candidate pairs from {target_tokenized_tsv}.")

## 4. Candidate Pairs Statistical Analysis & CSV Export
Analyze distribution of generated pairs across implicit and explicit combinations.

In [ ]:
if not pair_records:
    print("[peringatan] Tidak ada pasangan kandidat untuk dianalisis.")
    df_pairs = pd.DataFrame()
    rep.section("2. Hasil pembentukan pasangan").text(
        "Tidak ada pasangan kandidat. Jalankan notebook 02 lebih dulu agar "
        "`pred4pipeline.txt` tersedia."
    )
else:
    df_pairs = pd.DataFrame(pair_records)
    n = len(df_pairs)

    # Tabel 1: sumber & jumlah pasangan
    df_sumber = pd.DataFrame([{
        "Sumber_Prediksi": pred_file if pred_file else "(memakai file pair yang sudah ada)",
        "Total_Pasangan": n,
        "Kalimat_Unik": int(df_pairs["Sentence_ID"].nunique()),
        "Rata2_Pasangan_per_Kalimat": round(n / max(df_pairs["Sentence_ID"].nunique(), 1), 3),
        "File_Output": target_tokenized_tsv,
    }])
    rep.section("2. Hasil pembentukan pasangan")
    export_step_table(df_sumber, name="pair_01_sumber_dan_jumlah", csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Sumber & Jumlah Pasangan Kandidat ({DOMAIN.upper()})",
                      notes="Pasangan dibentuk dengan cross-product semua aspect x semua opinion per kalimat.")
    rep.table(df_sumber, caption="Ringkasan sumber")

    # Tabel 2: distribusi tipe pasangan
    pair_counts = df_pairs["Pair_Type"].value_counts()
    df_tipe = pair_counts.rename_axis("Tipe_Pasangan").reset_index(name="Jumlah")
    df_tipe["Persen"] = (df_tipe["Jumlah"] / n * 100).round(2)

    rep.section("3. Distribusi tipe pasangan")
    export_step_table(df_tipe, name="pair_02_distribusi_tipe", csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Distribusi Tipe Pasangan Implicit/Explicit ({DOMAIN.upper()})")
    rep.table(df_tipe, caption="Tipe pasangan")

    # Tabel 3: sebaran jumlah pasangan per kalimat
    per_sent = df_pairs.groupby("Sentence_ID").size()
    df_per_sent = per_sent.describe().to_frame("Nilai").reset_index()
    df_per_sent.columns = ["Statistik", "Nilai"]
    export_step_table(df_per_sent, name="pair_03_statistik_per_kalimat",
                      csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Statistik Jumlah Pasangan per Kalimat ({DOMAIN.upper()})")
    rep.table(df_per_sent, caption="Pasangan per kalimat")

    # Tabel 4: preview pasangan
    export_step_table(df_pairs.head(25), name="pair_04_preview", csv_dir=csv_dir, md_dir=md_dir,
                      title=f"Preview 25 Pasangan Kandidat ({DOMAIN.upper()})", max_rows_md=25)

    # CSV lengkap
    summary_csv = os.path.join(csv_dir, "candidate_pairs_summary.csv")
    df_pairs.to_csv(summary_csv, index=False, encoding="utf-8")
    print(f"[tabel] CSV lengkap semua pasangan: {summary_csv}")

    # Kalimat dengan ledakan pasangan terbanyak
    top_sent = (per_sent.sort_values(ascending=False).head(10)
                .rename_axis("Sentence_ID").reset_index(name="Jumlah_Pasangan"))
    top_sent = top_sent.merge(
        df_pairs.groupby("Sentence_ID")["Text"].first().reset_index(),
        on="Sentence_ID", how="left")
    top_sent["Text"] = top_sent["Text"].str.slice(0, 70)
    rep.section("4. Kalimat dengan pasangan terbanyak")
    export_step_table(top_sent, name="pair_05_kalimat_pasangan_terbanyak",
                      csv_dir=csv_dir, md_dir=md_dir,
                      title=f"10 Kalimat dengan Pasangan Kandidat Terbanyak ({DOMAIN.upper()})",
                      notes="Cross-product membuat kalimat dengan banyak span menghasilkan banyak kandidat, "
                            "sehingga menaikkan beban komputasi dan potensi false positive di step 2.")
    rep.table(top_sent, caption="Kalimat dengan kandidat terbanyak")


## 5. Visualization of Generated Candidate Pairs

In [ ]:
if df_pairs.empty:
    print("[plot] Dilewati: tidak ada data pasangan.")
else:
    n = len(df_pairs)
    pair_counts = df_pairs["Pair_Type"].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Kiri: distribusi tipe pasangan
    warna = ["#3498db", "#9b59b6", "#e67e22", "#e74c3c"][:len(pair_counts)]
    bars = axes[0].bar(range(len(pair_counts)), pair_counts.values,
                       color=warna, edgecolor="black", alpha=0.88)
    for b, v in zip(bars, pair_counts.values):
        axes[0].text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v/n*100:.1f}%)",
                     ha="center", va="bottom", fontsize=9, fontweight="bold")
    axes[0].set_xticks(range(len(pair_counts)))
    axes[0].set_xticklabels([t.replace("-", "\n") for t in pair_counts.index], fontsize=9)
    axes[0].set_title(f"[{DOMAIN.upper()}] Tipe Pasangan Kandidat", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Jumlah pasangan")
    axes[0].margins(y=0.18)
    axes[0].grid(axis="y", linestyle="--", alpha=0.7)

    # Kanan: histogram pasangan per kalimat
    per_sent = df_pairs.groupby("Sentence_ID").size()
    vc = per_sent.value_counts().sort_index()
    axes[1].bar(vc.index.astype(str), vc.values, color="#2ca02c", edgecolor="black", alpha=0.88)
    for x, v in zip(vc.index.astype(str), vc.values):
        axes[1].text(x, v, f"{v}", ha="center", va="bottom", fontsize=8, fontweight="bold")
    axes[1].set_title(f"[{DOMAIN.upper()}] Jumlah Pasangan per Kalimat", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Pasangan yang dihasilkan")
    axes[1].set_ylabel("Jumlah kalimat")
    axes[1].margins(y=0.15)
    axes[1].grid(axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout()
    plot_path = os.path.join(plots_dir, "04_candidate_pairs_distribution.png")
    plt.savefig(plot_path, dpi=300)
    plt.show()
    plt.close()
    print(f"[plot] Disimpan: {plot_path}")

    # Heatmap implicit aspect x implicit opinion
    pivot = pd.crosstab(
        df_pairs["Is_Implicit_Aspect"].map({True: "Implicit Aspect", False: "Explicit Aspect"}),
        df_pairs["Is_Implicit_Opinion"].map({True: "Implicit Opinion", False: "Explicit Opinion"}),
    )
    plt.figure(figsize=(7, 4.5))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5,
                cbar_kws={"label": "Jumlah pasangan"})
    plt.title(f"[{DOMAIN.upper()}] Matriks Implicit vs Explicit Pasangan Kandidat",
              fontsize=12, fontweight="bold")
    plt.ylabel("")
    plt.xlabel("")
    plt.tight_layout()
    heat_path = os.path.join(plots_dir, "04b_candidate_pairs_implicit_matrix.png")
    plt.savefig(heat_path, dpi=300)
    plt.show()
    plt.close()
    print(f"[plot] Disimpan: {heat_path}")

    rep.section("5. Visualisasi")
    rep.image(plot_path, "Distribusi tipe pasangan dan jumlah pasangan per kalimat")
    rep.image(heat_path, "Matriks implicit vs explicit pada pasangan kandidat")

rep.text(f"Sesi: `{active_session_dir}`")
report_path = rep.save()
print(f"\nLaporan Markdown step bridge: {report_path}")
print("Lanjut ke '04_ACOS_Step2_Category_Sentiment_Classification.ipynb'.")
